# ResNet-18: optimizer and regularization experiments

Adapted from `notebooks/_template/architecture/experiments.ipynb`: one architecture, seven explicit configurations (R1–R7), the shared `fit()` loop, comparison, conclusion and MLflow quality metrics.

All runs fine-tune ImageNet weights. The fixed v2 split has 513 train / 63 validation / 63 test clips. Checkpoints use the project's standard minimum validation EER; all 7 or 10 epochs finish.

The default topic `optimizers` reuses the original runs when available. Set `IML_TOPIC` to a new name before starting the kernel for a fresh experiment, or edit `TOPIC` below. Run cells in order. Outputs, weights and run artifacts stay local.


In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start the notebook inside the iml repository")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from iml.config import ExperimentConfig
from iml.cli import check_run_data, data_fingerprint, load_splits, record_run_data, summary_table
from iml.runs import RunRecord, run_dir, load_run
from iml.train import fit
from iml.tracking import Tracker, notebook_run

MEMBER = "petr"
MODEL = "resnet18"
TOPIC = os.environ.get("IML_TOPIC", "optimizers")
DATA = Path(os.environ.get("IML_DATA", str(ROOT / "data/training_dataset_v2"))).resolve()
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://localhost:5000")


## 1. Parameter configurations

`EXPLICIT` replaces the template's Cartesian grid because these were seven sequential trials. R1/R2 change optimizer; R2/R3 change dropout; R4/R5/R6 change weight decay; R5/R7 change learning rate. The 10-epoch and 7-epoch groups do not isolate epoch budget.


In [ ]:
BASE = ExperimentConfig(
    model=MODEL, member=MEMBER, topic=TOPIC,
    pretrained=True, batch_size=32, seed=42,
    scheduler="none", early_stopping_patience=0,
)
EXPLICIT = [
    ["epochs=10", "optimizer=sgd",   "lr=0.0001",  "dropout=0.2", "weight_decay=0.0"],
    ["epochs=10", "optimizer=adamw", "lr=0.0001",  "dropout=0.2", "weight_decay=0.0"],
    ["epochs=10", "optimizer=adamw", "lr=0.0001",  "dropout=0.4", "weight_decay=0.0"],
    ["epochs=7",  "optimizer=adamw", "lr=0.0001",  "dropout=0.1", "weight_decay=0.0"],
    ["epochs=7",  "optimizer=adamw", "lr=0.0001",  "dropout=0.1", "weight_decay=0.01"],
    ["epochs=7",  "optimizer=adamw", "lr=0.0001",  "dropout=0.1", "weight_decay=0.05"],
    ["epochs=7",  "optimizer=adamw", "lr=0.00005", "dropout=0.1", "weight_decay=0.01"],
]
CONFIGS = [BASE.with_overrides(*overrides) for overrides in EXPLICIT]
LABELS = [f"R{i}" for i in range(1, len(CONFIGS) + 1)]

def variant(cfg: ExperimentConfig) -> str:
    return ",".join(f"{k}={v}" for k, v in sorted(cfg.diff_from_default().items())) or "baseline"

pd.DataFrame([
    dict(run=label, optimizer=cfg.optimizer, lr=cfg.lr, dropout=cfg.dropout,
         weight_decay=cfg.weight_decay, epochs=cfg.epochs)
    for label, cfg in zip(LABELS, CONFIGS, strict=True)
])


## 2. Train or load finished runs

This is the template's cached loop: verify the data fingerprint, load finished runs, otherwise call the shared `fit()`. It never overwrites a finished run.

The original configs contain `checkpoint_metric="eer"`, a field removed from this PR. The notebook-only loader accepts and removes that redundant field **in memory**, rejects other values, and checks the loaded settings against the requested configuration. It does not modify configurations, weights or the shared loader.


In [ ]:
EXPECTED_SHA1 = "10ac1fb033e3b1845bc52ea2e2229c2982ad11f0"
FP = data_fingerprint(DATA)
if FP["sha1"] != EXPECTED_SHA1:
    raise ValueError("This report requires the original fixed training_dataset_v2 split")

def load_cached(out, expected):
    saved = json.loads((out / "config.json").read_text())
    if "checkpoint_metric" in saved:
        if saved.pop("checkpoint_metric") != "eer":
            raise ValueError("Expected the shared EER checkpoint protocol")
        record = RunRecord(
            path=out, cfg=ExperimentConfig.from_dict(saved),
            metrics=json.loads((out / "metrics.json").read_text()),
            history=pd.read_csv(out / "history.csv"),
        )
    else:
        record = load_run(out)
    if record.cfg.to_dict() != expected.to_dict():
        raise ValueError(f"Cached configuration differs: {out}")
    return record

records = []
for label, cfg in zip(LABELS, CONFIGS, strict=True):
    out = run_dir(cfg)
    check_run_data(out, FP, overwrite=False)
    if (out / "metrics.json").exists():
        print(label, "cached:", out.name)
        records.append(load_cached(out, cfg))
    else:
        print(label, "training:", out.name)
        records.append(fit(cfg, load_splits(cfg, data_dir=DATA), tracker=Tracker(enabled=False)))
        record_run_data(out, FP)


## 3. Compare: metrics and learning curves

The full shared summary stays in `table` for the template's MLflow cell. Only a compact subset is displayed. Curves use consistent run colors, saved-checkpoint circles and logarithmic loss axes, matching the report figures.


In [ ]:
table = summary_table(records)
label_by_name = {record.name: label for label, record in zip(LABELS, records, strict=True)}
compact = table[["run", "best_epoch", "val_eer", "test_accuracy", "test_far", "test_frr", "test_eer"]].copy()
compact["run"] = compact["run"].map(label_by_name)
for column in ("val_eer", "test_accuracy", "test_far", "test_frr", "test_eer"):
    compact[column] = (compact[column] * 100).round(2)
compact = compact.rename(columns={column: column + " (%)" for column in compact if column not in ("run", "best_epoch")})
compact.sort_values("run").reset_index(drop=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

def plot_report_curves(records, out):
    plt.rcParams.update({'font.size':11, 'axes.spines.top':False, 'axes.spines.right':False})
    for name, rs, title in [
        ('optimizers_dropout', records[:3], 'ResNet-18 fine-tuning: optimizer and dropout trials (10 epochs)'),
        ('regularization_lr', records[3:], 'ResNet-18 fine-tuning: regularization and learning-rate trials (7 epochs)'),
    ]:
        fig, axes = plt.subplots(1, 3, figsize=(16, 6))
        handles, labels = [], []
        for r, color, marker in zip(rs, ['#0072b2','#d55e00','#009e73','#aa4499'], ['o','s','^','D']):
            h, c, m = r['history'], r['config'], r['metrics']
            for ax, col, scale in zip(axes, ['train_loss','val_loss','val_eer'], [1,1,100]):
                line, = ax.plot(h.epoch, h[col]*scale, color=color, marker=marker, ms=4, lw=1.7)
                best = h.loc[h.epoch == m['best_epoch']].iloc[0]
                ax.scatter(best.epoch, best[col]*scale, s=120, facecolors='none', edgecolors=color, linewidths=1.6, zorder=5)
            handles.append(line)
            labels.append(f"{r['id']}: {c['optimizer'].upper()}, LR={c['lr']:g}, dropout={c['dropout']:g}, WD={c['weight_decay']:g}; saved epoch {m['best_epoch']}")
        losses = np.concatenate([r['history'][['train_loss','val_loss']].to_numpy().ravel() for r in rs])
        for ax in axes[:2]:
            ax.set_yscale('log')
            ax.set_ylim(losses.min()/1.5, losses.max()*1.5)
            ax.set_ylabel('Cross-entropy (log scale)')
        for ax, t in zip(axes, ['Training loss', 'Validation loss', 'Validation EER']):
            ax.set_title(t)
            ax.set_xlabel('Epoch')
            ax.set_xticks(range(1, rs[0]['config']['epochs']+1))
            ax.grid(alpha=.22, which='major')
        axes[2].set_ylabel('EER (%)')
        axes[2].set_ylim(-2, max(r['history'].val_eer.max()*100 for r in rs)+3)
        fig.suptitle(title, fontsize=15, y=.98)
        fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(.5,.005), ncol=1, fontsize=10, frameon=False)
        fig.subplots_adjust(left=.065,right=.985,top=.86,bottom=.30,wspace=.30)
        fig.savefig(out/'plots'/f'{name}.png',dpi=180,facecolor='white')
        plt.close(fig)

out = run_dir(CONFIGS[0]).parent / "comparison"
(out / "plots").mkdir(parents=True, exist_ok=True)
table.to_csv(out / "table.csv", index=False)
plot_records = [
    dict(id=label, config=cfg.to_dict(), metrics=record.metrics, history=record.history)
    for label, cfg, record in zip(LABELS, CONFIGS, records, strict=True)
]
plot_report_curves(plot_records, out)
display(Image(out / "plots/optimizers_dropout.png"), Image(out / "plots/regularization_lr.png"))


### Conclusion

- R1/R2: AdamW converged faster at the same LR, reducing test false rejects from 6 to 3 but increasing false accepts from 4 to 5.
- R2/R3: higher dropout reduced false accepts from 5 to 2 but increased false rejects from 3 to 6; accuracy stayed unchanged.
- R4/R5/R6: WD 0.01 gave 7 test errors versus 9 for both WD 0 and WD 0.05.
- R5/R7: halving LR reduced test errors from 7 to 6. R7 had the highest observed accuracy, 90.48% (57/63): 2 false accepts and 4 false rejects, at its validation-selected threshold. Its saved checkpoint is epoch 5 of 7.

R7 is an exploratory candidate, not an independently validated winner: these are one-seed results on a small test set inspected between trials. Validation/test differences remain unexplained. T's seven validation clips come from one source recording. Zero validation EER is not itself a bug; nonzero EER is not an objective. No causal claim about microphone memorization or removal of overfitting follows from these trials.


## 4. MLflow (optional; quality metrics only)

Run the following cell only when you want to log the comparison to your MLflow server. It retains the template's behavior: one run per variant, classification-quality metrics and parameters only; no figures, tables or artifacts. Without a reachable server the project helper warns.


In [ ]:
import re

QUALITY = re.compile(r"^(val|test)_(accuracy|far|frr|eer|macro_f1|f1_macro)$")


def _param(v):
    return str(v) if isinstance(v, (list, tuple, dict)) else v


rows = table.set_index("run").to_dict("index")
for cfg, r in zip(CONFIGS, records):
    row = rows[r.name]
    metrics = {k: float(v) for k, v in row.items() if QUALITY.match(k) and pd.notna(v)}
    params = {"model": MODEL, **{k: _param(v) for k, v in cfg.diff_from_default().items()}}
    tags = {"variant": variant(cfg)}
    with notebook_run(member=MEMBER, topic=TOPIC, run_name=r.name, params=params, tags=tags) as run:
        run.log_metrics(metrics)